<a href="https://colab.research.google.com/github/2025cschandanapb-ai/DAV/blob/main/MINI_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")

print("Libraries imported successfully!")

!wget -q https://archive.ics.uci.edu/static/public/320/student+performance.zip -O student.zip
!unzip -o -q student.zip

df = pd.read_csv("student-mat.csv", sep=";")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

In [ ]:
print("First five records:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nStatistical description:")
display(df.describe())

print("\nNumber of rows and columns:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate records:")
print(df.duplicated().sum())

In [ ]:
# Remove duplicate records
df = df.drop_duplicates()

# Check and handle missing values
print("Missing values before cleaning:")
print(df.isnull().sum().sum())

# Fill numerical missing values with median
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with mode
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values after cleaning:")
print(df.isnull().sum().sum())

print("\nCleaned dataset shape:", df.shape)

In [ ]:
print("STATISTICAL ANALYSIS")

print("\nMean of final grades:")
print(df['G3'].mean())

print("\nMedian of final grades:")
print(df['G3'].median())

print("\nMode of final grades:")
print(df['G3'].mode()[0])

print("\nVariance of final grades:")
print(df['G3'].var())

print("\nStandard deviation of final grades:")
print(df['G3'].std())

print("\nMinimum marks:", df['G3'].min())
print("Maximum marks:", df['G3'].max())

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df, x='sex')

plt.title("Student Distribution by Gender")
plt.xlabel("Gender")
plt.ylabel("Number of Students")

plt.show()

In [ ]:
plt.figure(figsize=(7,4))

sns.barplot(data=df, x='studytime', y='G3', errorbar=None)

plt.title("Study Time vs Average Final Marks")
plt.xlabel("Weekly Study Time")
plt.ylabel("Average Final Marks")

plt.show()

In [ ]:
plt.figure(figsize=(7,4))

sns.histplot(df['G3'], bins=10, kde=True)

plt.title("Distribution of Student Final Grades")
plt.xlabel("Final Grade")
plt.ylabel("Number of Students")

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

sns.boxplot(data=df, x='sex', y='G3')

plt.title("Final Grades Based on Gender")
plt.xlabel("Gender")
plt.ylabel("Final Grade")

plt.show()

In [ ]:
plt.figure(figsize=(12,8))

numeric_df = df.select_dtypes(include=np.number)

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title("Correlation Heatmap of Student Performance")
plt.show()

In [ ]:
plt.figure(figsize=(12,8))

numeric_df = df.select_dtypes(include=np.number)

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title("Correlation Heatmap of Student Performance")
plt.show()

In [ ]:
plt.figure(figsize=(7,4))

sns.scatterplot(data=df, x='absences', y='G3')

plt.title("School Absences vs Final Grades")
plt.xlabel("Number of Absences")
plt.ylabel("Final Grade")

plt.show()

In [ ]:
def performance_category(mark):
    if mark < 10:
        return "Low"
    elif mark < 14:
        return "Medium"
    else:
        return "High"

df['Performance'] = df['G3'].apply(performance_category)

print(df['Performance'].value_counts())

plt.figure(figsize=(6,4))

sns.countplot(data=df, x='Performance',
              order=['Low', 'Medium', 'High'])

plt.title("Student Performance Categories")
plt.xlabel("Performance")
plt.ylabel("Number of Students")

plt.show()

In [ ]:
# Input features
X = df.drop(columns=['G1', 'G2', 'G3', 'Performance'])

# Target variable
y = df['G3']

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include='object'
).columns.tolist()

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'),
         categorical_features)
    ]
)

# Random Forest model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

# Train model
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

print("Model trained successfully!")

In [ ]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("MODEL EVALUATION RESULTS")

print("Mean Absolute Error:", round(mae, 2))
print("Mean Squared Error:", round(mse, 2))
print("Root Mean Squared Error:", round(rmse, 2))
print("R2 Score:", round(r2, 2))

In [ ]:
plt.figure(figsize=(7,5))

plt.scatter(y_test, y_pred)

plt.plot(
    [0, 20],
    [0, 20],
    linestyle='--'
)

plt.title("Actual vs Predicted Student Marks")
plt.xlabel("Actual Marks")
plt.ylabel("Predicted Marks")

plt.show()

In [ ]:
feature_names = model.named_steps[
    'preprocessor'
].get_feature_names_out()

importance = model.named_steps[
    'regressor'
].feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

print(importance_df.head(10))

plt.figure(figsize=(9,5))

sns.barplot(
    data=importance_df.head(10),
    x='Importance',
    y='Feature'
)

plt.title("Top 10 Important Features")
plt.xlabel("Importance")
plt.ylabel("Features")

plt.show()

In [ ]:
!pip -q install ipywidgets plotly

import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, HTML

# Dashboard title
print("STUDENT PERFORMANCE ANALYSIS DASHBOARD")

# Interactive filters
gender_filter = widgets.Dropdown(
    options=['All'] + sorted(df['sex'].unique().tolist()),
    value='All',
    description='Gender:'
)

study_filter = widgets.Dropdown(
    options=['All'] + sorted(df['studytime'].unique().tolist()),
    value='All',
    description='Study Time:'
)

def create_dashboard(gender, studytime):

    filtered = df.copy()

    if gender != 'All':
        filtered = filtered[filtered['sex'] == gender]

    if studytime != 'All':
        filtered = filtered[filtered['studytime'] == studytime]

    if len(filtered) == 0:
        display(HTML("<h3>No students found for these filters.</h3>"))
        return

    # Calculate KPIs
    total = len(filtered)
    average = filtered['G3'].mean()
    highest = filtered['G3'].max()
    lowest = filtered['G3'].min()
    pass_percentage = (filtered['G3'] >= 10).mean() * 100

    # KPI display
    display(HTML(f"""
    <div style="display:flex; flex-wrap:wrap; gap:12px;">

      <div style="background:#dbeafe;padding:15px;border-radius:10px;min-width:120px">
        <h4>Total Students</h4>
        <h2>{total}</h2>
      </div>

      <div style="background:#dcfce7;padding:15px;border-radius:10px;min-width:120px">
        <h4>Average Marks</h4>
        <h2>{average:.2f}</h2>
      </div>

      <div style="background:#fef3c7;padding:15px;border-radius:10px;min-width:120px">
        <h4>Highest Marks</h4>
        <h2>{highest}</h2>
      </div>

      <div style="background:#fee2e2;padding:15px;border-radius:10px;min-width:120px">
        <h4>Lowest Marks</h4>
        <h2>{lowest}</h2>
      </div>

      <div style="background:#ede9fe;padding:15px;border-radius:10px;min-width:120px">
        <h4>Pass Percentage</h4>
        <h2>{pass_percentage:.2f}%</h2>
      </div>

    </div>
    """))

    # Chart 1: Gender distribution
    fig1 = px.pie(
        filtered,
        names='sex',
        title='Student Gender Distribution',
        hole=0.3
    )
    fig1.show()

    # Chart 2: Study time vs marks
    study_data = filtered.groupby(
        'studytime', as_index=False
    )['G3'].mean()

    fig2 = px.bar(
        study_data,
        x='studytime',
        y='G3',
        title='Study Time vs Average Marks',
        labels={
            'studytime': 'Study Time Level',
            'G3': 'Average Final Marks'
        }
    )
    fig2.show()

    # Chart 3: Performance categories
    fig3 = px.histogram(
        filtered,
        x='Performance',
        color='Performance',
        title='Student Performance Categories',
        category_orders={
            'Performance': ['Low', 'Medium', 'High']
        }
    )
    fig3.show()

    # Chart 4: Absences vs marks
    fig4 = px.scatter(
        filtered,
        x='absences',
        y='G3',
        color='sex',
        title='Absences vs Final Marks',
        labels={
            'absences': 'Number of Absences',
            'G3': 'Final Marks'
        }
    )
    fig4.show()

    # Display key insights
    print("\nKEY INSIGHTS")
    print("Total students:", total)
    print("Average marks:", round(average, 2))
    print("Highest marks:", highest)
    print("Pass percentage:", round(pass_percentage, 2), "%")

    print(
        "Most common performance category:",
        filtered['Performance'].mode()[0]
    )

# Connect filters to dashboard
dashboard = widgets.interactive_output(
    create_dashboard,
    {
        'gender': gender_filter,
        'studytime': study_filter
    }
)

display(widgets.VBox([
    widgets.HTML("<h2>Student Performance Dashboard</h2>"),
    gender_filter,
    study_filter,
    dashboard
]))

In [ ]:
# Select one student from testing data
sample_student = X_test.iloc[[0]]

# Predict final marks
predicted_marks = model.predict(sample_student)[0]

actual_marks = y_test.iloc[0]

print("STUDENT PERFORMANCE PREDICTION")

print("Actual Marks:", actual_marks)
print("Predicted Marks:", round(predicted_marks, 2))

if predicted_marks < 10:
    print("Predicted Category: Low")
elif predicted_marks < 14:
    print("Predicted Category: Medium")
else:
    print("Predicted Category: High")